# v101_name_frequency — v001 + core-name frequency features

| Field | Value |
|---|---|
| **Version** | `v101_name_frequency` |
| **Plan group** | C5 (integrated run, INT) |
| **Parent version** | v001 |
| **Author** | M1 rajaguru2004 |
| **Date** | 2026-09-25 |
| **Status** | shortlisted |

v001 plus one feature group; every other stage and parameter is as in v001. Every stage is library code in
`src/entity_resolution/` (tested, documented); this notebook only configures, runs and
inspects it:

```
raw TSV -> normalize (static rules + learned transliteration map) -> blocking (exact keys,
name+address word TF-IDF, short-address name char-grams, per country) -> features (47 pair
similarities, chunked) -> LightGBM matcher -> decision rule tuned for macro F0.5 on the tune
split (pool-side 1-to-1) -> matching_results.tsv + candidate_pairs.tsv
```

House rules: every code cell is preceded by a markdown cell saying what it does and why;
the only decision metric is macro F0.5 on the fixed validation fold.

## 1. Hypothesis

* **Change vs parent (v001):** one new feature group, `frequency` (six features): how many
  S1 records and how many pool records share each side's core name (per million records of
  the country, counted over the whole fold, never the training sample), how common the S1's
  first token is in the pool, and core-name equality.
* **Why it should raise macro F0.5:** in the v001 dry run the largest loss was recall
  (0.015 of 0.022), and the typical miss was an exact-name pool record with an empty
  address scored ~0.05, while typical false merges were same-name decoys of common names.
  The matcher could not tell a rare name (safe to match on the name alone) from a common one
  (needs address evidence). Frequencies give it exactly that.
* **Expected effect:** fewer misses on unique names with empty addresses, fewer false merges on
  common names; val macro F0.5 above v001 by > 0.002 with harder-val not lower.
* **Config fix carried along:** v001's LightGBM stopped at its 2,000-round cap with the tune
  log-loss still falling (early stopping never triggered), so the cap is raised to 4,000;
  early stopping (100 rounds on the tune sample) still decides the length.
* **Discard if:** val F0.5 ≤ v001 + 0.002, or harder-val falls.

## 2. Setup

Imports from the shared library, this experiment's folders and the pipeline configuration.
`PipelineConfig()` holds every choice of this version (its defaults *are* the V1 plan);
it is saved to `artifacts/config.json` and logged in `metrics.json`. `timings` collects the
stage run times under the eight standard labels (13 §2.2).

In [1]:
import json
import subprocess
import sys
import time
from dataclasses import asdict

import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.blocking import PASS_BITS
from entity_resolution.evaluate import error_samples, harder_fold, pair_in, slice_report
from entity_resolution.features import FEATURE_COLUMNS, feature_names
from entity_resolution.normalize import NORM_COLUMNS, normalise_records
from entity_resolution.pipeline import (
    PipelineConfig, fit, load_normalised, peak_rss_gb, run_fold, run_test,
)
from entity_resolution.split import load_fold
from entity_resolution.tracking import log_result, timed

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", 30)

EXP_DIR = C.EXPERIMENTS / "v101_name_frequency"
ARTIFACTS = EXP_DIR / "artifacts"
from entity_resolution.features import DEFAULT_GROUPS
from entity_resolution.model import MatcherParams
cfg = PipelineConfig(feature_groups=(*DEFAULT_GROUPS, "frequency"),
                     model=MatcherParams(n_estimators=4000))
timings: dict[str, float] = {}
t_start = time.time()
print(json.dumps(cfg.record(), indent=1)[:3000])

{
 "normalise": {
  "transliterate": true,
  "strip_legal": true,
  "expand_abbrev": true,
  "region_map": null,
  "chunk_rows": 1000000,
  "learn_token_map": true,
  "token_map_min_count": 3,
  "token_map_min_share": 0.5
 },
 "blocking": {
  "exact_keys": [
   "name_core",
   "name_sorted",
   "name_squash"
  ],
  "exact_max_group": 50,
  "name_char": {
   "column": "name_core",
   "analyzer": "char_wb",
   "ngram": [
    3,
    3
   ],
   "top_k": 10,
   "min_sim": 0.5,
   "max_df": 0.2,
   "min_df": 2,
   "sublinear_tf": true,
   "pool_max_addr_tokens": 3,
   "max_df_abs": 20000
  },
  "name_addr_word": {
   "column": "name_addr",
   "analyzer": "word",
   "ngram": [
    1,
    2
   ],
   "top_k": 25,
   "min_sim": 0.2,
   "max_df": 0.01,
   "min_df": 2,
   "sublinear_tf": true,
   "pool_max_addr_tokens": null,
   "max_df_abs": 10000
  },
  "addr_char": null,
  "max_per_s1": 60,
  "s1_chunk": 50000,
  "n_threads": 12,
  "vocab_sample": 500000,
  "seed": 42
 },
 "feature_groups": [
 

## 3. Data

The fixed validation split (`split.load_fold`, 20 % of Source 1 by id hash, seed 42): every
version scores the same held-out entities against the same pool, so local F0.5 values are
comparable. Everything trainable is fitted on the `train` fold only; inside it,
`trainset.inner_split` (seed 4242, 25 % tune) separates model training (fit side) from
early stopping and threshold tuning (tune side). The val pool keeps the records matched to
*other* val entities: they are the decoys that make singletons and same-name businesses hard.

In [2]:
with timed("load", timings):
    train = load_fold("train", columns=[])   # ids only: the pipeline reads normalised records
    val = load_fold("val")                   # raw columns kept for the error samples
pd.DataFrame([train.summary(), val.summary()])

,fold,s1,s2,s3,true_pairs,singleton_share
0,train,1765488,4026279,4229875,6110753,0.0558
1,val,441333,1008337,1055728,1527612,0.0559


## 4. Method

### 4.1 Normalisation (`normalize.py`, `token_maps.py`)

Rules, in order (05 §2), each undoing a noise pattern measured on true pairs:

* **Transliteration** (`anyascii`, ISC): rows with non-ASCII characters are transliterated on
  the raw text (Indic scripts, accents). 7 % of pool names and 9 % of pool addresses are in an
  Indic script; Source 1 is all Latin.
* **Case, `&` → `and`, punctuation, apostrophes, dotted initials** (`L.L.C.` → `llc`).
* **Domain / handle forms** (`allh0spitalityproducts.com`, `@sakashpoint`,
  `ORTHOPEDICHEALTHCOM`) lose their suffix; **leet digits** inside words fold to letters
  (`F0rman` → `forman`).
* **Legal forms** leave `name_core` and are kept, canonical, in `legal_form` (`Pvt. Ltd.` =
  `Private Limited` = `pvt ltd`; SARL/SAS/EURL for France). **Honorific prefixes** injected
  into pool names (`Mr`, `Smt`, `Shri`) are dropped.
* **Learned transliteration map**: for true pairs of the train fold whose pool name is in an
  Indic script, tokens are aligned by position with the Latin name (`सॉल्यूशंस` →
  `solyusms` ↔ `solutions`); tokens seen ≥ 3 times with a ≥ 50 % consistent partner form a
  map applied to non-Latin names. Learned from the provided training pairs only.
* **Addresses**: region components (full name, code or native-script name: `Maharashtra`,
  `MH`, `महाराष्ट्र`) become one code; ordinals lose their suffix; digits split from letters;
  leading zeros go; street types are canonicalised (`Street`/`St`/`Saint` — the generator
  writes `Saint` for `St`); old/new Indian city names are unified.

Derived keys: `name_first`, `name_sorted` (sorted distinct tokens), `name_squash` (letters and
digits only), `addr_nums`, `postcode`, `region`, `name_addr`. The cell shows raw and
normalised val records, including Indic-script names (static rules only: the learned map is
fitted inside `pipeline.fit` in §4.4, which prints examples of it).

In [3]:
examples = pd.concat([val.s2[val.s2[C.NAME].str.contains(r"[\x{0900}-\x{0DFF}]", regex=True)].head(3),
                      val.s3.sample(5, random_state=1)], ignore_index=True)
normalise_records(examples)[["entity_id", "name_norm", "name_core", "legal_form", "name_squash",
                             "addr_norm", "region"]].assign(raw_name=examples[C.NAME].to_numpy())

,entity_id,name_norm,name_core,legal_form,name_squash,addr_norm,region,raw_name
0,S2-322802571,hotl vemcrs limited,hotl vemcrs,ltd,hotlvemcrs,lucknow c 121 meena bakery chauraha nr dariyapur up,up,होटल वेंचर्स लिमिटेड
1,S2-669353485,lksmi devlprs praivet limited,lksmi devlprs,pvt ltd,lksmidevlprs,h no 1338 nesari tal gadhinglaj kolhapur kolhapur mh,mh,लक्ष्मी डेवलपर्स प्राइवेट लिमिटेड
2,S2-938256871,daynamik indastrij praibhet limited,daynamik indastrij,pvt ltd,daynamikindastrij,78 a raja ram mohan roy sarani howrah calcutta wb,wb,ডায়নামিক ইন্ডাস্ট্রিজ প্রাইভেট লিমিটেড
3,S3-782716519,trading peacock health exports private limited,trading peacock health exports,pvt ltd,tradingpeacockhealthexports,no 14 fl chennai tn,tn,Trading Peacock Health Exports Private Limited
4,S3-990078959,cascade dynamic bumrungrad corporation,cascade dynamic bumrungrad,corp,cascadedynamicbumrungrad,frankfort 1060 1 2 clmnton st in,in,Cascade Dynamic Bumrungrad Corporation
5,S3-42890805,arista magnetics private limited,arista magnetics,pvt ltd,aristamagnetics,om kameswari chennai w mambalam tn,tn,Arista Magnetics Private-Limited
6,S3-897685301,kestial partners,kestial partners,,kestialpartners,8254 seneca turnpike clinton ny,ny,... Kestial Partners
7,S3-395399434,janys premier publishing,janys premier publishing,,janyspremierpublishing,twin lakes dr orange tx,tx,Jany'S Prémier Publishing


### 4.2 Blocking (`blocking.py`)

Candidate pairs are generated inside each country partition (whatever country values exist,
so France needs nothing special) as the union of:

| Bit | Pass | Why |
|---|---|---|
| 1 / 2 / 4 | exact `name_core` / `name_sorted` / `name_squash`, pool key groups ≤ 50 | cheap, exact; word swaps; domain/leet forms |
| 8 | name char 3-gram TF-IDF top-10 (cos ≥ 0.5) against pool records with ≤ 3 address tokens | typos where the address cannot help (empty / city-only addresses) |
| 16 | name + address word uni+bigram TF-IDF top-25 (cos ≥ 0.2, `max_df` 0.01) | the workhorse: renames, script names, same-name decoys ranked by their address |

Pairs are capped at 60 per S1 (exact pairs first, then by similarity). Retrieval uses
`sparse_dot_topn` (Apache-2.0) multi-threaded sparse top-k. Measured on 10k-S1 val samples
while designing it: running char 3-grams on every record costs ~3 ms per S1 (hours on test),
so that pass is restricted; word bigrams keep the address signal that `max_df` removes from
frequent unigrams (`rajendra nagar`, `5 52`). Recall is reported on the full val fold in §5.

In [4]:
pd.Series({k: v for k, v in asdict(cfg.blocking).items()})

exact_keys                               (name_core, name_sorted, name_squash)
exact_max_group                                                             50
name_char          {'column': 'name_core', 'analyzer': 'char_wb', 'ngram': ...
name_addr_word     {'column': 'name_addr', 'analyzer': 'word', 'ngram': (1,...
addr_char                                                                 None
max_per_s1                                                                  60
s1_chunk                                                                 50000
n_threads                                                                   12
vocab_sample                                                            500000
seed                                                                        42
dtype: object

### 4.3 Pair features (`features.py`)

Similarity features per candidate pair, grouped as in 07 §2. Names rank, addresses decide:
many address-agreement features (token set / Jaccard / containment, house-number and number
set agreement, region, last tokens) sit next to fuzzy name scores (rapidfuzz, MIT) on the
normalised and core names, legal-form agreement, blocking similarities and the pair's
context inside its S1 group (rank and gap to the best candidate). No country feature (open
set). Every similarity is in [0, 1]; NaN where a field is empty (LightGBM handles it).

In [5]:
pd.DataFrame([(g, ", ".join(FEATURE_COLUMNS[g])) for g in cfg.feature_groups],
             columns=["group", "features"]).assign(n=lambda d: d.features.str.count(",") + 1)

,group,features,n
0,blocking,"pass_exact, pass_name_char, pass_name_addr, sim_name_cha...",6
1,name_fuzzy,"nm_ratio, nm_partial, nm_token_sort, nm_token_set, nm_jw...",10
2,name_tokens,"tok_jaccard, tok_dice, tok_common, tok_len_l, tok_len_r,...",8
3,legal,"legal_eq, legal_missing_l, legal_missing_r",3
4,numeric,"num_jaccard, num_shared_any, num_first_eq, postcode_eq",4
5,address,"ad_token_set, ad_partial, ad_ratio, ad_jaccard, ad_conta...",8
6,context,"ctx_rank_name, ctx_gap_name, ctx_rank_addr, ctx_gap_addr...",5
7,meta,"is_s3, non_latin_r, len_ratio_name",3
8,frequency,"fq_s1_l, fq_pool_l, fq_first_pool_l, fq_pool_r, fq_s1_r,...",6


### 4.4 Matching model and 4.5 decision rule (`model.py`, `decision.py`, `pipeline.fit`)

* **Training pairs**: 200k S1 entities sampled from the fit side of the inner split, each with
  all its candidates against the whole fit pool (so decoy density is realistic); label 1 for
  true pairs. Truth pairs outside the candidates cannot be learned; they are counted by the
  metric.
* **LightGBM** (MIT): binary log-loss, 63 leaves, learning rate 0.05, `min_data_in_leaf` 200,
  feature/bagging fraction 0.8, early stopping (100 rounds) on a 50k-S1 sample of the tune side;
  deterministic, 12 threads. No class reweighting: precision is bought in the decision
  layer, never by distorting probabilities.
* **Decision rule**: per S1 entity, pool-side 1-to-1 first (a pool record is kept only for its
  highest-probability S1 entity: every pool record belongs to at most one S1 in the training
  data), then keep pairs with `prob ≥ tau_abs`, `prob ≥ tau_rel · p_max`, entity emptied when
  `p_max < tau_single`, at most `max_matches`. The four thresholds are grid-searched (3,906
  rules plus a refinement) for macro F0.5 on **all** tune-side S1 entities, scored like
  inference, so singletons, blocking misses and 1-to-1 competition are all accounted for.
  Ties go to the most conservative rule (the test pool holds more decoys than train).

The cell runs the whole fit: normalisation cache, token map, blocking of the fit / stop /
tune sides, features, LightGBM, scoring of the tune side and the rule grid. The val fold is
not touched.

In [6]:
fit_timings: dict[str, float] = {}
t0 = time.time()
fitted = fit(cfg, train, ARTIFACTS, fit_timings)
timings["fit_seconds"] = fit_timings.get("fit_seconds", 0.0)
timings["tune_seconds"] = fit_timings.get("tune_seconds", 0.0)
print(f"fit() total {time.time() - t0:.0f} s; stages: {fit_timings}")
print("token map:", fitted.info["token_map_size"], "tokens, e.g.",
      list(fitted.token_map.items())[:12])
print("training:", {k: v for k, v in fitted.info["fit_info"].items()})
print("rule:", fitted.rule)

fit() total 716 s; stages: {'normalise_seconds': 4.96, 'fit_load_seconds': 25.17, 'fit_blocking_seconds': 5.63, 'stop_load_seconds': 9.48, 'stop_blocking_seconds': 1.64, 'features_seconds': 52.71, 'fit_seconds': 343.51, 'tune_load_seconds': 10.79, 'tune_blocking_seconds': 3.44, 'score_seconds': 226.73, 'tune_seconds': 14.99}
token map: 536 tokens, e.g. [('aditia', 'aditya'), ('adity', 'aditya'), ('aigro', 'agro'), ('aiksports', 'exports'), ('ailailpi', 'llp'), ('aimtrpraijij', 'enterprises'), ('ainrji', 'energy'), ('aisais', 'ss'), ('aiti', 'it'), ('akro', 'agro'), ('alkpa', 'alpha'), ('alph', 'alpha')]
training: {'rows': 6719931, 'positive_rate': 0.09974239318826339, 'best_iteration': 1666, 'tune_logloss': 0.00939002092230011, 'tune_auc': 0.9998572234168527, 'fit_seconds': 343.51}
rule: DecisionRule(tau_abs=0.42, tau_rel=0.0, tau_single=0.52, max_matches=11, one_to_one=True)


Blocking quality of the three training-time sides (recall, candidates per S1), the 20 most
important features (gain, normalised) and the best rules of the tune grid. An address
feature missing from the top 15 would be a bug (08 §8).

In [7]:
blk = pd.DataFrame({side: fitted.info[f"{side}_blocking"] for side in ("fit", "stop", "tune")}).T
display(blk[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean", "candidates_p95"]])
display(fitted.matcher.importance().head(20).rename("gain").to_frame())
fitted.tune_table.sort_values("f_beta", ascending=False).head(10)

,pair_recall,entity_recall,ceiling_f_beta,candidates_mean,candidates_p95
fit,0.973580,0.997445,0.990903,33.764425,42.0
stop,0.990733,0.999216,0.997052,32.803258,60.0
tune,0.990317,0.999275,0.996959,32.963458,60.0


,gain
feature,
ad_token_set,0.404484
sim_name_addr_word,0.161029
core_token_set,0.061086
ctx_rank_addr,0.054993
num_jaccard,0.052303
ad_jaccard,0.037740
core_jw,0.029525
nm_token_sort,0.027285
squash_ratio,0.021418


,tau_abs,tau_rel,tau_single,max_matches,one_to_one,f_beta,n_pred,pair_precision,pair_recall,match_rate,stage
386,0.36,0.0,0.46,11,True,0.986226,1483546,0.995964,0.967702,0.942809,grid
260,0.34,0.0,0.44,11,True,0.986218,1485043,0.995689,0.968411,0.942924,grid
383,0.36,0.0,0.41,11,True,0.986202,1483687,0.995916,0.967747,0.943108,grid
509,0.38,0.0,0.43,11,True,0.986196,1482226,0.996164,0.967035,0.942986,grid
635,0.40,0.0,0.45,11,True,0.986196,1480712,0.996421,0.966296,0.942868,grid
512,0.38,0.0,0.48,11,True,0.986185,1482096,0.996203,0.966989,0.942705,grid
761,0.42,0.0,0.47,11,True,0.986180,1479248,0.996652,0.965565,0.942768,grid
263,0.34,0.0,0.49,11,True,0.986178,1484923,0.995721,0.968364,0.942666,grid
257,0.34,0.0,0.39,11,True,0.986175,1485173,0.995640,0.968449,0.943208,grid
389,0.36,0.0,0.51,11,True,0.986174,1483448,0.995988,0.967662,0.942589,grid


## 5. Evaluation on the validation fold

`run_fold` blocks, scores and decides the val fold once with the frozen rule. Primary metric:
**macro F0.5 over all val S1 entities, singletons included** (`evaluate.score_pairs`, equal to
`metrics.breakdown`). Blocking quality: candidate pair recall, entity recall and the ceiling
F0.5 a perfect matcher would reach on these candidates.

In [8]:
t0 = time.time()
metrics, val_pairs, val_scored, val_matches = run_fold(cfg, fitted, val)
for k in ("blocking_seconds", "score_seconds", "decide_seconds", "normalise_seconds"):
    timings[k] = metrics.get(k, 0.0)
print(f"run_fold {time.time() - t0:.0f} s")
pd.Series(metrics)

run_fold 257 s


f_beta                    0.985822
f_beta_singletons         0.987401
f_beta_matched            0.985729
pair_precision            0.996570
pair_recall               0.965428
entities             441333.000000
singletons            24684.000000
cand_recall               0.990570
entity_recall             0.999330
ceiling_f_beta            0.997067
cands_mean               32.970535
cands_p95                60.000000
normalise_seconds        10.090000
blocking_seconds          1.500000
score_seconds           233.630000
decide_seconds            6.710000
dtype: float64

Recall of each blocking pass on the val fold (a pair can come from several passes) and the
standard slice report (11 §7): country, source, Indic-script pool names, ambiguous core
names, singletons, number of true matches, empty addresses.

In [9]:
is_true = pair_in(val_pairs, val.pairs)          # candidate pair is a true pair
rows = []
for name, bit in PASS_BITS.items():
    in_pass = (val_pairs["pass"].to_numpy() & bit) != 0
    if in_pass.any():
        rows.append((name, in_pass.sum() / len(val.s1), (in_pass & is_true).sum() / len(val.pairs)))
display(pd.DataFrame(rows, columns=["pass", "pairs_per_s1", "recall"]))
s1n_val = load_normalised("train", (1,), cfg, val.s1[C.ENTITY_ID], fitted.token_map)
pooln_val = load_normalised("train", (2, 3), cfg, pd.concat([val.s2, val.s3])[C.ENTITY_ID],
                            fitted.token_map)
slices = slice_report(val_matches, val, s1n_val, pooln_val)
slices.to_csv(ARTIFACTS / "slices.csv", index=False)
slices

,pass,pairs_per_s1,recall
0,exact_core,7.244736,0.564582
1,exact_sorted,7.333775,0.595844
2,exact_squash,7.311683,0.594464
3,name_char,7.387163,0.045524
4,name_addr_word,21.615297,0.982539


,family,slice,entities,f_beta,pair_precision,pair_recall,n_true,n_pred,tp
0,country,India,176522,0.985025,0.996239,0.963036,611167,590798,588576
1,country,US,264811,0.986354,0.996790,0.967023,916445,889077,886223
2,country,all,441333,0.985822,0.996570,0.965428,1527612,1479875,1474799
3,source,S2,384063,0.977925,0.996637,0.967896,739443,718119,715704
4,source,S3,388275,0.975063,0.996507,0.963112,788169,761756,759095
5,non_latin,yes,54312,0.986264,0.998348,0.956551,204745,196173,195849
6,non_latin,no,387021,0.985761,0.996298,0.966802,1322867,1283702,1278950
7,non_latin,all,441333,0.985822,0.996570,0.965428,1527612,1479875,1474799
8,domain_form,yes,82591,0.987887,0.997216,0.965912,349356,338389,337447
9,domain_form,no,358742,0.985347,0.996378,0.965284,1178256,1141486,1137352


**Harder validation** (11 §6): test has 5.8 pool records per S1 against 4.7 in train, so the
same frozen rule is also scored on a val variant that drops 20 % of the S1 entities but keeps
their pool records (they become unowned decoys). A version whose harder score falls while
val rises is buying recall with false merges.

In [10]:
harder_metrics, *_ = run_fold(cfg, fitted, harder_fold(val), tag="harder")
print({k: round(v, 4) for k, v in harder_metrics.items() if k.startswith("f_beta") or k.startswith("pair")})

{'f_beta': 0.9852, 'f_beta_singletons': 0.9848, 'f_beta_matched': 0.9852, 'pair_precision': 0.9955, 'pair_recall': 0.9659}


## 6. Error analysis

Samples of the four error kinds, both records side by side with the model probability:
false merges on matched entities, missed true pairs, matched entities predicted empty
(false singletons) and predictions on true singletons. The counts say where the lost F0.5
sits; the samples name the pattern for the next version.

In [11]:
counts = {}
for kind in ("false_merge", "missed", "false_singleton", "singleton_merge"):
    sample = error_samples(val_matches, val, kind, n=10, scored=val_scored)
    counts[kind] = len(error_samples(val_matches, val, kind, n=10**9))
    print(f"--- {kind}: {counts[kind]} pairs")
    display(sample)

--- false_merge: 4757 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-137695892,S3-697508841,0.436653,Classic Suisse LLC,"Phoenix, 15801 48th Street, AZ, Unit 1216",Classic Suisse LLC,
1,S1-166811225,S2-152577270,0.675499,"Cascio, Smith & Drew","16-17 163 Street, Whitestone, NY","LLC Cascio, Smith & Drew","16-19 163 ST, WHITESTONE CITY, NY"
2,S1-193648498,S2-676218258,0.890639,Chamunda Brothers,"Bangalore North, Karnataka, 12Th Main, 3Rd Phase, Peenya...",Chamunda Brothers Private Limited,"NO.205/B-8A, 12TH MAIN, 3RD PHASE, PEENYA INDUSTRIAL ARE..."
3,S1-227041204,S2-929028816,0.539311,GYY Estates Private Limited,"53/4D, 22Nd Cross Someshwara Layout, Bannerghatta Road, ...",GY LIMITED PRIVATE ESTATES,"53/4D., 22ND CROSS SOMESHWARA LAYOUT, BANNERGHATTA ROAD,..."
4,S1-245365691,S2-462500961,0.448225,"Cindy M. Baca, P.A.","184 Money Lane, Winchester, TN","Cindy M. Baca, P.C. Services",
5,S1-507502381,S2-390328928,0.438986,Mridul Traders Private Limited,"No 264, Akshaya Complex, 1St Floore, 8Th Block, Nagarabh...",MRIDUL PRIVATE LIMITED-CENTER,
6,S1-608927025,S2-802800679,0.718729,Optimal Mobility Group,"11 Wildwood Drive, Newburgh, NY",Optimal Mobility Group LLC,"14 WILDWOOD DR, NEWBURGH, NY"
7,S1-734849917,S3-835178366,0.503154,"Frontier Utility, LLC","Chesapeake City, Unit F, 301 Wimbledon Chase, VA","Frontier Útility, LLC",
8,S1-738128729,S2-959360977,0.985643,Faclara Capital Partners LLC,"8 Web Road, Georgetown, MA",Faclara Capital Partners,"29 WEB ROAD, MA, GEORGETOWN"
9,S1-856275644,S3-441555318,0.579729,Cunningham Holding Company Inc,"1296 Cortez Street, Grants, NM",Cunningham Holding Company Holdings,"1307 Cortez St, PO Box 4056, Grants, New Mexico"


--- missed: 51267 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-165098328,S3-923481956,0.206423,Kaur Global Digital Inc,"115 Mckinley Avenue, Sapulpa, OK",Vantageyuma Sys,"115 Mckinley Avenue, Sapulpa, OK"
1,S1-422860972,S3-398485332,0.259933,Osborne & Schalk,"5325 Mandarin Circle, Chattanooga, TN",Osborne & Schalk Corp | www.osborne.com,"532 Mandarin Circle, Chattanooga, Tennessee"
2,S1-470476768,S2-434918813,0.380532,B 3 Certified Disciplined,"400 Janet Street, Tahlequah, OK",B CERTIFIED DISCIPLINED CENTER,
3,S1-479462718,S3-844680400,0.099252,Memorial Fellowship LLC,"109 Turkey Hollow Road, Campbell County, VA",The Memorial Fellowship LLC,
4,S1-585940696,S3-706624556,0.111922,Capital Alphabet,"25 Hunting Hollow Drive, Pepper Pike, OH",Calovera Labs,"25 Hunting Hollow Drive, Pepper Pike, OH"
5,S1-634305339,S2-131572980,0.098779,Universal Exports Private Limited,"C-1, G-11, Ground Floor, Krishna Apra Plaza, Sector Alph...",यूनिवर्सल एक्सपोर्ट्स प्राइवेट लिमिटेड,"C-##1, LUCKNOW HQ REGION, उत्तर प्रदेश"
6,S1-711377995,S3-750418846,0.395118,Digital Hospitality Limited,"261/3, Prince Anwar Shah Road, Kolkata, Kolkata, Howrah,...",Digital Limited Services #66970,"0598 261/3, Prince Anwar Shah Road, Kolkata, Kolkata, WB"
7,S1-718326235,S3-740299382,NaN,HHD Beverages Private Limited,"1/25, Hazuri Bhawan, Peepal Mandi Road, Agra, Uttar Pradesh",Ariapyra,"Agra, 1/25, UP, Agra"
8,S1-892913472,S2-937795220,0.152755,Best Beverage Holdings,"6833 Walnut Avenue, Orangevale, CA",Best Beverage,
9,S1-98315586,S2-315720146,0.123842,Delta Homecare,"1225 Osteen Street, Unit 1, Vidor, TX",Umbraectoorbi,"1225 OSTEEN ST, VIDOR, TX"


--- false_singleton: 1546 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-111857209,S2-238473467,NaN,Ymc India Limited,"D-632, Tower-7, Ashiana Upvan, Ahinsa Khand-2, Indirapur...",Services Ymc Limited,"D-63, GHAZIABAD, INDIRAPURAM, Uttar Pradesh"
1,S1-202273665,S2-741550226,0.487043,Mumbai Assurance Partners,"Pl:178, Ft No:11, Flr No 2, Ashok Apt, Shradhanand Rd, M...",MUMBAI ASSURANCE PARTNERS LIMITED,"PL:778, FT NO:11, FLR NO 2, ASHOK APT, SHRADHANAND RD, M..."
2,S1-305224497,S2-544426290,NaN,Varois,"N3937 Schielke Road, Town Of Schley, WI",SOLZETA,"SCHIELKE RD, GLEASOON, WI"
3,S1-373915344,S2-517883037,0.418769,Bush & Durrett Chemical,"185 Quincy Shore Drive, Unit A67, Quincy, MA",Dugerott & Bush Chemical,"196 QUINCY SHORE DRIVE, QUINCY, MA"
4,S1-433616795,S3-300908171,NaN,Cox Grand Paper Inc,"39 Sobro Avenue, Hempstead, NY",Cox Grand Pmep Inc,"#39 Sobro Avenue, Valley Stream, New York"
5,S1-60374765,S2-705571307,0.087760,Straight Edge Auto Body,"727 Millers Road, Des Plaines, IL",STRAIGHT EDGE AUTO BODY INC.,"997- MILLERS RD, DES PLAINES, IL"
6,S1-663224990,S2-244465048,0.301205,Coimbatore Marketing Group,"1/684 F, Sathi Main Road Kunnathur, Annur, Coimbatore, T...",Coimbatore Group-Center,"1677 F, SATHI MAIN ROAD KUNNATHUR, ANNUR, COIMBATORE, Ta..."
7,S1-741906218,S3-528175808,0.353287,Caps Suspensions (India) Private Limited,"H.No.113 Ahmedgarh Road, Pohir, Ludhiana, Punjab",Caps Suspensions (India) Pbrddhvadte Limited,"PB, H.no.13 Ahmedagrh Road, Ludhiana"
8,S1-919213174,S3-411199990,0.371734,Poonam Jewelery Private Limited,"C-63, Surya Nagar, Ghaziabad, Uttar Pradesh",Poonam Jewe1ery Pridnnate Limited,"C-62, Surya Nagar, Ghaziabad, UP"
9,S1-982251971,S3-887774250,0.000670,Shiv Projects,"640, Block-O New Alipore, Kolkata, Kolkata, Howrah, West...",Shiv Projects,"40, Kolkota, পশ্চিমবঙ্গ"


--- singleton_merge: 319 pairs


,source1_entity_id,entity_id,prob,name_l,addr_l,name_r,addr_r
0,S1-181116332,S2-765632443,0.971265,Smyrna Animal Hospital Inc.,"110 Creek Court, Smyrna, TN",Smyrna Animal Hospital Inc,"TN, SMYRNA, 114 CREEK CT"
1,S1-189228639,S3-977832728,0.727249,Management Db Farmer Private Limited,"Planner House C 21/87A, Mahamandal Nagar, Lahurabir, Var...",Management Db Impex Private Limited,"Planner House C 21/100A, Varanasi, UP"
2,S1-232548372,S2-410743734,0.782644,Garmendia & Villanueva Solution LLC,"2710 Rio Linda Boulevard, Sacramento, CA",Evoecto,"2671 RIO LINDA BLVD, SACRAMENTO, CA"
3,S1-30056058,S2-957804806,0.996156,Osprey Group,"231 Silvermine Avenue, Norwalk, CT",Osprey Group,"232 Silvermine Ave, NORWALK, CT"
4,S1-318775017,S2-175392776,0.640355,Domjur Systems Ltd,"77/5/6, Benaras Road, Domjur, Howrah, West Bengal",Domjur Systems Infratech Limited - 3987383648,"77/5/27, BENARAS ROAD, DOMJUR, West Bengal"
5,S1-433391156,S3-930848067,0.632450,International Products Private Limited,"#40, 1St Floor, 6Th Main Road, Jp Nagar 3Rd Phase, Banga...",LLP International Products,"KA, Bangalore, 1St Floor, 6Th Main Road, Jp Nagar 3Rd Ph..."
6,S1-622878224,S2-208978463,0.973183,CZZ Solana Inc.,"MA, Franklin, 137 Union Street",CZZ Solana,
7,S1-718172744,S3-460238385,0.987935,Visoft Capital LLC,"4807 Cowslip Court, Oxon Hill, MD",Visoft Capital Partners,"4807 Cowslip Ct, Maryland, Oxon Hill"
8,S1-782452787,S2-206216664,0.642549,Smith Equity Partners LLC,"1116 7th Street, Havre, MT",SMITH EQUITY PARTNERS PARTNERS,"001121 SEVENTH ST, HAVRE, MT"
9,S1-982362384,S3-883430624,0.663909,NX Neer Ltd,"13, Satyanarayan Temple Road Salkia, Howrah, West Bengal",NU Néer Ltd,"13, Satyanarayan Temple Road Salkia, Howrah, WB"


## 7. Log the result

Records `metrics.json` and this version's row in `experiments/experiments.csv`, stamped
with the git commit of `src/` (it must not end in `-dirty`). Keys follow 13 §2.2.

In [12]:
record = {
    "hypothesis": "core-name frequency features cut misses on unique names and false merges on common names",
    "blocking_config": asdict(cfg.blocking), "feature_groups": list(cfg.feature_groups),
    "model_params": asdict(cfg.model), "rule": asdict(fitted.rule),
    **{k: metrics[k] for k in ("f_beta", "f_beta_singletons", "f_beta_matched",
                               "pair_precision", "pair_recall")},
    **{k: metrics[k] for k in ("cand_recall", "entity_recall", "ceiling_f_beta",
                               "cands_mean", "cands_p95")},
    "harder_f_beta": harder_metrics["f_beta"],
    "tune_f_beta": float(fitted.tune_table["f_beta"].max()),
    "n_fp": counts["false_merge"] + counts["singleton_merge"], "n_fn": counts["missed"],
    "n_false_singleton": counts["false_singleton"], "errors": counts,
    "token_map_size": fitted.info["token_map_size"],
    "best_iteration": fitted.matcher.best_iteration_,
    **timings, "peak_rss_gb": peak_rss_gb(),
}
parent = json.loads((C.EXPERIMENTS / "v001_base_model" / "metrics.json").read_text())["metrics"]
DECISION = ("KEEP" if record["f_beta"] > parent["f_beta"] + 0.002
            and record["harder_f_beta"] >= parent["harder_f_beta"] else "DROP")
record["decision"] = DECISION
print("parent v001:", parent["f_beta"], "->", record["f_beta"], DECISION)
row = log_result(
    EXP_DIR, change="v001 + core-name frequency features; LightGBM cap 4000 rounds",
    group="C5", local_f05=metrics["f_beta"], cand_recall=metrics["cand_recall"],
    notes=(f"cands {metrics['cands_mean']:.1f}/S1; singleton F0.5 "
           f"{metrics['f_beta_singletons']:.4f}; harder {harder_metrics['f_beta']:.4f}"),
    metrics=record, owner="M1", parent="v001", decision=DECISION)
row

parent v001: 0.9843647201357917 -> 0.9858224874376977 DROP


{'version': 'v101',
 'date': '2026-09-25',
 'group': 'C5',
 'change': 'v001 + core-name frequency features; LightGBM cap 4000 rounds',
 'local_f05': '0.9858',
 'cand_recall': '0.9906',
 'public_f05': '',
 'commit': '4133d2b',
 'notes': 'cands 33.0/S1; singleton F0.5 0.9874; harder 0.9852',
 'owner': 'M1',
 'parent': 'v001',
 'decision': 'DROP'}

## 8. Conclusion

Written after the run from the numbers above (see the markdown cell at the end of §9).

## 9. Test inference (shortlisted: upload #1)

Same pipeline on the test split: normalise (cached), block per country (France included),
score, decide with the frozen rule, write both files with `submission.write_pairs` from the
exact pairs frame that was scored. Then the sanity checks of 11 §10: one row per test S1 in
both files, every country present with candidates and matches, match rates and candidates per
S1 close to val.

In [13]:
t0 = time.time()
match_path, cand_path, s1n_test, test_matches, test_summary = run_test(cfg, fitted)
print(f"run_test {time.time() - t0:.0f} s -> {match_path}, {cand_path}")


def per_country(s1n, matches, n_cands_by_s1):
    """Match rate, matches and candidates per S1, by country (11 §10 sanity table)."""
    country = s1n.set_index(C.ENTITY_ID)[C.COUNTRY]
    n_s1 = s1n.groupby(C.COUNTRY).size()
    by = matches[C.S1_ID].map(country)
    return pd.DataFrame({
        "s1": n_s1,
        "cands_per_s1": n_cands_by_s1.groupby(n_cands_by_s1.index.map(country)).sum() / n_s1,
        "matched_share": matches.groupby(by)[C.S1_ID].nunique() / n_s1,
        "matches_per_s1": matches.groupby(by).size() / n_s1,
    })


test_table = per_country(s1n_test, test_matches, test_summary["n_cands"])
val_table = per_country(s1n_val, val_matches, val_pairs.groupby(C.S1_ID).size())
display(pd.concat({"test": test_table, "val": val_table}))
country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
pd.crosstab(test_summary.index.map(country_of),
            pd.cut(test_summary["p_max"], [0, .1, .3, .5, .7, .9, 1.0]), normalize="index").round(3)

run_test 1107 s -> /home/suryaguru/StudioProjects/aws/business_entity_resolution/output/matching_results.tsv, /home/suryaguru/StudioProjects/aws/business_entity_resolution/output/candidate_pairs.tsv


s1  cands_per_s1  matched_share  matches_per_s1
test France  259452     37.044401       0.948572        3.400968
     India   809986     34.960182       0.940400        3.307602
     US      663106     34.072592       0.942501        3.361906
val  India   176522     33.666064       0.941888        3.346880
     US      264811     32.506897       0.942129        3.357402

p_max,"(0.0, 0.1]","(0.1, 0.3]","(0.3, 0.5]","(0.5, 0.7]","(0.7, 0.9]","(0.9, 1.0]"
row_0,,,,,,
France,0.022,0.015,0.008,0.005,0.007,0.944
India,0.043,0.010,0.005,0.004,0.004,0.934
US,0.039,0.013,0.004,0.003,0.003,0.938


Both validators on the exact files that will be uploaded: ours (`submission.validate` with
id existence checks) and the organisers' stdlib-only `validate_submission.py`.

In [14]:
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-2000:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")

PASS
 


ML Challenge 2026 — submission validator
  test dir: /home/suryaguru/StudioProjects/aws/business_entity_resolution/dataset/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (99746 empty, 1632798 non-empty).
  candidate_pairs.tsv: 1732544 rows (0 empty, 1732544 non-empty).

PASS — no blocking issues found. Safe to submit.
 
notebook total 2429 s, peak RSS 10.14 GB
